## Preface

The goal of this script is to build out the logic for how the break up and generation of vcf files will work. We will start with the generation of the vcf file from the sequencing files. I will be using the conda environment in this logic script, but the final product will be built around an apptainer and a pyqt interface.

The assumed working directory is the jobs folder.

In [1]:
import pandas as pd
import subprocess as sp
import gzip as gz
import shutil as shu
import itertools as it
import os as os
import numpy as np
import math as math
import collections as collections
import sys as sys
from scipy import stats

## Initiate Environmental Variables

In [2]:
gtf_file = "../raw_data/I.hed-braker-rna-orthodb.gtf"
vcf_file = "../raw_data/Ihed_filtered_biallelic_SNP.vcf.gz"

memory_saving = True
batch_size = 1000000

## Sequencing Quality Control

We have raw fastq files. We need to do some preprocessing before we can use them to build the vcf files. One important thing to keep in mind is that we may need to do hard-clipping too.

## NOTES

Because of the time crunch for practicing for this interview, we will use the actual vcf and gtf file for processing.

## GTF Processing

In [3]:
#get the columns with information that are not attributes
n_columns = 8
gtf = pd.read_table(gtf_file, header = None, usecols = range(n_columns), sep = "\t")
gtf.columns = ["scaffold", "prediction_source", "feature", "start", "end", "score", "strand", "frame"]

#the reason for separating the attributes is that sometimes people include apostrophes or hashtags which mess with reading in as a pandas dataframe
gtf_attr = []
attr_index = 8
with open(gtf_file, "r") as file:
    gtf_attr = [line.split("\t")[attr_index].strip() for line in file]

#does your dataframe make sense?
gtf.info()

<class 'pandas.DataFrame'>
RangeIndex: 698273 entries, 0 to 698272
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   scaffold           698273 non-null  str  
 1   prediction_source  698273 non-null  str  
 2   feature            698273 non-null  str  
 3   start              698273 non-null  int64
 4   end                698273 non-null  int64
 5   score              698273 non-null  str  
 6   strand             698273 non-null  str  
 7   frame              698273 non-null  str  
dtypes: int64(2), str(6)
memory usage: 42.6 MB


## VCF Processing

In [4]:
#uncompress vcf
#with gz.open(vcf_file, "rb") as f_in:
#    with open(vcf_file[0:-3], "wb") as f_out:
#        shu.copyfileobj(f_in, f_out)

#peek into vcf structure
#basically we want to know what information is included in the vcf
#os.system("head " + vcf_file[0:-3] + " -n 65 | tail -n 10")
os.system("tail " + vcf_file[0:-3] + " -n 2")
#the vcf data is starting at line 64 (index 63) with the column names starting with a hashtag

Chr15	33618496	.	A	T	27153.3	PASS	DP=2286;VDB=0.544298;SGB=216.749;RPBZ=-2.24466;MQBZ=-0.821404;MQSBZ=-0.454422;BQBZ=-0.605783;SCBZ=-0.181357;MQ0F=0.00262467;AC=212;AN=242;DP4=165,170,960,957;MQ=57;F_MISSING=0.0162602;NS=121;AF=0.876033;MAF=0.123967;AC_Het=0;AC_Hom=212;AC_Hemi=0;HWE=2.85791e-20;ExcHet=1	GT:PL:DP:AD:GQ:VAF:VAF1	1/1:243,27,0:9:0,9:32:1:1	1/1:255,30,0:10:0,10:35:1:1	1/1:129,12,0:4:0,4:17:1:1	1/1:212,27,0:9:0,9:32:1:1	1/1:255,39,0:13:0,13:44:1:1	1/1:133,15,0:5:0,5:20:1:1	1/1:182,18,0:6:0,6:23:1:1	1/1:255,45,0:15:0,15:50:1:1	0/0:0,21,169:7:7,0:10:0:0	1/1:255,35,0:16:1,15:40:0.9375:0.9375	./.:0,0,0:0:0,0:0:0:0	0/0:0,72,255:24:24,0:60:0:0	1/1:255,39,0:13:0,13:44:1:1	1/1:255,66,0:22:0,22:71:1:1	1/1:255,33,0:11:0,11:38:1:1	0/0:0,42,255:14:14,0:30:0:0	1/1:255,72,0:24:0,24:77:1:1	1/1:255,69,0:23:0,23:74:1:1	1/1:255,48,0:16:0,16:53:1:1	1/1:182,18,0:6:0,6:23:1:1	1/1:209,48,0:16:0,16:53:1:1	1/1:255,42,0:14:0,14:47:1:1	1/1:255,60,0:20:0,20:65:1:1	1/1:255,48,0:16:0,16:53:1:1	1/1:255,7

0

5,0:25:0,25:80:1:1	1/1:255,66,0:22:0,22:71:1:1	1/1:195,21,0:7:0,7:26:1:1	1/1:255,57,0:19:0,19:62:1:1	1/1:255,66,0:22:0,22:71:1:1	0/0:0,87,255:29:29,0:75:0:0	1/1:255,57,0:19:0,19:62:1:1	1/1:255,39,0:13:0,13:44:1:1	1/1:255,66,0:22:0,22:71:1:1	1/1:255,84,0:28:0,28:89:1:1	0/0:0,21,181:7:7,0:10:0:0	0/0:0,114,255:38:38,0:102:0:0	0/0:0,72,255:24:24,0:60:0:0	1/1:255,48,0:17:0,16:53:1:1	1/1:255,66,0:22:0,22:71:1:1	0/0:0,60,255:20:20,0:48:0:0	1/1:255,60,0:20:0,20:65:1:1	1/1:255,65,0:31:2,29:70:0.935484:0.935484	1/1:235,33,0:11:0,11:38:1:1	1/1:255,75,0:25:0,25:80:1:1	1/1:255,51,0:17:0,17:56:1:1	1/1:255,69,0:23:0,23:74:1:1	1/1:255,48,0:16:0,16:53:1:1	1/1:249,36,0:12:0,12:41:1:1	1/1:255,54,0:18:0,18:59:1:1	1/1:255,63,0:21:0,21:68:1:1	1/1:255,87,0:29:0,29:92:1:1	1/1:255,39,0:13:0,13:44:1:1	0/0:0,87,255:29:29,0:75:0:0	1/1:255,51,0:17:0,17:56:1:1	1/1:255,42,0:14:0,14:47:1:1	1/1:248,26,0:13:1,12:31:0.923077:0.923077	1/1:255,81,0:27:0,27:86:1:1	1/1:255,45,0:15:0,15:50:1:1	1/1:255,51,0:17:0,17:56:1:1	1/1

In [5]:
#let's split the vcf file into separate files that we can simplify their structure
header_line = 63
header = ""
with open(vcf_file[0:-3]) as file:
    for i, line in enumerate(file):
        if i == header_line:
            header = line
        elif i <= header_line:
            pass
        else:
            break

#let's get rid of the hashtag and convert to a comma separated list
header_list = header.split("\t")
header_list = [i.strip() for i in header_list] #get rid of flanking spaces and new lines
header_list = [i.strip("#") for i in header_list] #get rid of hashtags

In [6]:
header_list

#create column headers for selecting relevant data
snp_meta = header_list[0:7]
snp_info_col = 7
format_info_col = 8
genotypes = header_list[9:None]

In [7]:
#select columns that have common/useful metadata for SNPs
snp_metadata = pd.read_table(vcf_file[0:-3], sep = "\t", header = None, 
                             usecols = range(0, 7), skiprows = header_line + 1, names = snp_meta)

#we then want a way to map the genotype info to a format. we really only need one entry for this
with open(vcf_file[0:-3], "r") as file:
    format_info = next(it.islice(file, header_line + 1, header_line + 2), None)
format_info = format_info.split("\t")[format_info_col]

#genotype information is really where the bulk of the memory overload is. For most analyses, we really just need the genotype information.
#So, instead of reading in the whole genotype info as a pandas dataframe, it may make more sense to iterate through and just extract the only info we need
geno_matrix = []
geno_dict = {"0/0":0, "0/1":1, "1/1":2, "./.":9}
if memory_saving == False:
    with open(vcf_file[0:-3], "r") as file:
        for i, line in enumerate(file):
            if i <= header_line:
                pass
            else:
                #iteratively go through lines
                genotype_info = line.strip().split("\t")
                genotype_info = genotype_info[9:None]
                first_vals = []
                for j in genotype_info:
                    first_vals.append(j.split(":")[0])
                first_vals = list(map(lambda item: geno_dict.get(item), first_vals))
                #first_vals = [geno_dict.get(item) for item in first_vals]
                geno_matrix.append(first_vals)
        print(geno_matrix)
        geno_matrix = np.array(geno_matrix)
elif memory_saving == True:
    with open(vcf_file[0:-3], "r") as file:
        for i, line in enumerate(file):
            if i <= header_line:
                pass
            elif i > header_line and (i % batch_size != 0):
                #iteratively go through lines
                genotype_info = line.strip().split("\t")
                genotype_info = genotype_info[9:None]
                first_vals = []
                for j in genotype_info:
                    first_vals.append(j.split(":")[0])
                first_vals = list(map(lambda item: geno_dict.get(item), first_vals))
                #first_vals = [geno_dict.get(item) for item in first_vals]
                geno_matrix.append(first_vals)
                if  i > header_line and i == (snp_metadata.shape[0] + header_line):
                    geno_matrix = np.array(geno_matrix)
                    print(geno_matrix)
                    print("Saving at line " + str(i) + " because memory_saving set to True")
                    np.savetxt("../processed_data/vcf_geno_" + str(i) + ".geno", geno_matrix, delimiter = "", fmt = "%d")
                    geno_matrix = []
            elif i > header_line and (i % batch_size == 0):
                #iteratively go through lines
                genotype_info = line.strip().split("\t")
                genotype_info = genotype_info[9:None]
                first_vals = []
                for j in genotype_info:
                    first_vals.append(j.split(":")[0])
                first_vals = list(map(lambda item: geno_dict.get(item), first_vals))
                #first_vals = [geno_dict.get(item) for item in first_vals]
                geno_matrix.append(first_vals)
                geno_matrix = np.array(geno_matrix)
                print(geno_matrix)
                print("Saving at line " + str(i) + " because memory_saving set to True")
                np.savetxt("../processed_data/vcf_geno_" + str(i) + ".geno", geno_matrix, delimiter = "", fmt = "%d")
                geno_matrix = []
else:
    print("memory_saving variable not set.")

[[0 0 0 ... 0 0 0]
 [0 1 0 ... 1 1 1]
 [0 1 0 ... 1 0 1]
 ...
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]]
Saving at line 1000000 because memory_saving set to True
[[0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]]
Saving at line 2000000 because memory_saving set to True
[[0 2 0 ... 0 0 0]
 [0 2 0 ... 0 0 0]
 [0 2 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [2 2 2 ... 2 0 0]
 [0 0 0 ... 0 0 0]]
Saving at line 2990384 because memory_saving set to True


In [8]:
#concatenate genotype matrix files and delete temporary ones
#os.system("cat ../processed_data/vcf_geno_*.geno > ../processed_data/combined_vcf_geno.geno")
#os.system("rm ../processed_data/vcf_geno_*.geno")

#calculate minor allele frequency and missing genotype frequencyfrom genotype matrix
maf = []
missing_freq = []
with open("../processed_data/combined_vcf_geno.geno", "r") as file:
    for i, line in enumerate(file):
        geno = list(line.strip())
        geno_count = collections.Counter(geno)
        allele_freqs = [(geno_count["0"] + 0.5 * geno_count["1"])/(geno_count["0"] + geno_count["1"] + geno_count["2"]),
                        (geno_count["2"] + 0.5 * geno_count["1"])/(geno_count["0"] + geno_count["1"] + geno_count["2"])]
        maf.append(min(allele_freqs))

        missing_freq.append(geno_count["9"]/len(geno))

In [34]:
#can we read in the genotype matrix as an individual file?
geno_matrix = []
with open("../processed_data/combined_vcf_geno.geno", "r") as file:
    for i, line in enumerate(file):
        #the last line is the new line character we need to remove
        geno = list(line.strip())
        geno_matrix.append(geno)

geno_matrix = np.array(geno_matrix, dtype = np.int16)

In [36]:
##let's remove multiallelic SNPs because a lot of imputation software can't handle these kinds of sites
#how many multiallelic SNPs are there?
site_counts = snp_metadata.groupby(["CHROM", "POS"]).size()
count_freq = site_counts.value_counts()

#if there are multiallelic SNPs, exit and print error message
if len(count_freq) > 1:
    print("ERROR: Multiallelic SNPs detected. Please filter vcf file to only include biallelic SNPs. Frequency of allele counts is included below:")
    print(count_freq)
    sys.exit()
else:
    print("All Clear for Imputation: All sites are biallelic.")

All Clear for Imputation: All sites are biallelic.


In [37]:
#run imputation
imputation_options = ["mode", "beagle"]
imp_strat = "mode"
if imp_strat == "mode":
    #get mode of rows in geno matrix
    geno_mode = stats.mode(geno_matrix, axis = 1)
    geno_mode = np.asarray(geno_mode)[0]
    
    #create mask of where missing values are
    geno_mask = geno_matrix == 9
    rows, cols = np.where(geno_mask)
    
    #mutate array with indexing
    geno_imp = geno_matrix
    geno_imp[rows, cols] = geno_mode[rows]
elif imp_strat == "beagle":
    pass
else:
    print("ERROR: Imputation strategy is neither 'mode' nor 'beagle'. Please rerun with one of these options selected.")
    sys.exit()

np.int64(0)